# Modèle complet linéaire

Les variables du modèle sont les grandes masses du compte de résultat:
- produits : contributions industrielles (MAD, cash et assimilés), dotation France 2030 socle, projets collaboratifs, prestations, autres revenus
- charges : masse salariale, achats, sous-traitance, coûts fixes structure, amortissements, frais fianciers, provisions

Les variables varient peu autour d'un point de fonctionnement choisi (par exemple année N-1).

Le mix d'activité est supposé **fixe** : répartition entre projets de recherche en propre, prestations et projets collaboratifs; répartition entre RF,RI,DE,FS dans les projets de recherche en propre.

Ces hypothèses permettent d'avoir un modèle linéaire, valable au voisinage du point de fonctionnement.

In [1]:
# import librairies
import pyomo.environ as pyo

# create model
m = pyo.ConcreteModel()

# Vue compte de résultat (plan comptable)

Infos 22/10
- masse salariale IRT 2026          = 20.7 M€
- achats projets                    =  8.8 M€ (6 M€ ANR, 1.5 M€ prestations)
- achats hors projets indirects     =  6.0 M€
- MAD                               =  3.0 M€ (projets), 1.0 M€ (hors projets)
- cash                              = ?

CA 2026
- 36.0 M€ = 9.6 M€ F2030 + contrib membres 13.8 M€ + 3e tiers 12.5 M€ (FILAE + prestations + HE)

## Produits comptables

### Projets de recherche en propre

In [2]:
# MAD facturée aux partenaires (unités = M€) (NB : >=0) pour les projets de recherche en propre
m.mad = pyo.Var(bounds=(3.5, 4.5))  

# contribution partenaires en cash (et assimilé) (unités = M€) pour les projets de recherche en propre
m.cash = pyo.Var(bounds=(9.0, 11.0)) 

# calcul des cofinancements privés
m.cofinancements_prives = pyo.Expression(
    expr=m.mad+m.cash
)

# France 2030 socle, cappé à 11M€ par Convention 2026.
m.f2030_max = pyo.Param(initialize=11.0)

# dotation France 2030 effective
m.f2030 = pyo.Var(bounds=(0.0, m.f2030_max)) 

### Projets collaboratifs

Horizon Europe, BPI, AID, CNES... tout type d'appel d'offre public ou assimilé

In [3]:
m.projets_collaboratifs = pyo.Var(bounds=(0.0, 2.0))  

### Prestations

In [4]:
m.prestations = pyo.Var(bounds=(0.0, 8.0))

### Autres sources de revenus

Autres subventions (Région, Département...), revenus financiers, etc.

In [5]:
m.autres_revenus = pyo.Var(bounds=(4.0, 6.0))

### Total produits

In [6]:
m.produits = pyo.Expression(
    expr=m.cofinancements_prives + m.f2030 + m.projets_collaboratifs + m.prestations + m.autres_revenus
)

## Charges comptables

### Masse salariale

- C'est la masse salariale chargée, **au sens du plan comptable**
- On ne distingue pas à ce stade de directs, indirects, etc.
- Cela arrive dans la vue comptabilité analytique plus loin

In [7]:
m.masse_salariale = pyo.Var(bounds=(20.5, 21.0))  # Hypothèse

### Achats et sous-traitance

In [8]:
# cumul achats et sous-traitance pour toutes activités
# distingo rendu nécessaires par règle des 50% max sous-traitance sur projets

m.x_achats = pyo.Var(bounds=(4.0, 5.0))   # unités en M€. VALEUR ?
m.x_st = pyo.Var(bounds=(4.0, 5.0))   # unités en M€. VALEUR ?

### MAD

In [9]:
# MAD facturée par les partenaires
# => il s'agit (au signe près) de m.mad qui apparaît en produits plus haut (double facturation)

### Autres charges

In [10]:
# autres coûts fixes de structure (locaux, électricité, etc.)
m.structure = pyo.Param(initialize=6.0)  # A REVOIR AVEC UNE VRAIE ESTIMATION

# amortissements (charge calculée)
m.amortissements = pyo.Param(initialize=0.0)   # VRAIE VALEUR ?

# frais financiers, impôts, provisions
m.frais_fi = pyo.Param(initialize=0.0)   # VRAIE VALEUR ?
m.provisions = pyo.Param(initialize=0.0)   # VRAIE VALEUR ?

### Total charges

In [11]:
m.charges = pyo.Expression(
    expr=m.masse_salariale + m.x_achats + m.x_st + m.mad + m.structure + m.amortissements + m.frais_fi + m.provisions
)

### Résultat comptable

In [12]:
m.resultat = pyo.Expression(
    expr=m.produits - m.charges
)

# Répartition analytique

Dans un premier temps, le mix d'activité est un paramètre (par exemple les ratios historiques), pas une variable. 

Ce mix s'applique aux coûts directs. Les coûts d'environnement et les financements associés s'en déduisent.

Cela permet de rester à un modèle linéaire. 

Si on veut faire varier le mix, le modèle devient non linéaire (degré 3) et nécessite un solveur non linéaire.

### Mix activités

In [13]:
# mix entre projets de recherche en propre (PRP), projets collaboratifs, prestations
m.beta_prp = pyo.Param(initialize=0.70)
m.beta_projets_collaboratifs = pyo.Param(initialize=0.05)
m.beta_prestations = pyo.Param(initialize=0.25)

# check somme égale à 1.0..
assert m.beta_prp + m.beta_projets_collaboratifs + m.beta_prestations == 1.0, "Le mix d'activités ne somme pas à 1.0"

### Mix projets recherche en propre

- les revenus associés aux PRP dépendent des coûts complets associés via les différentes intensités de financement.
- les coûts complets incluent des frais d'environnement calculés sur l'assiette des coûts admissibles par des règles ANR
- la clef de répartition de base est un mix de projets des différents types (RF,RI,DE,FS) (FS pour faisabilité) correspondant à différentes intensités
- cette clef est utilisée pour la ventilation en coûts directs

In [14]:
# ratios historiques
m.prp_rf = pyo.Param(initialize=0.05)
m.prp_ri = pyo.Param(initialize=0.95)
m.prp_de = pyo.Param(initialize=0.00)
m.prp_fs = pyo.Param(initialize=0.00)

# check somme égale à 1.0..
assert m.prp_rf+m.prp_ri+m.prp_de+m.prp_fs == 1.0, "Le mix d'activités projets recherche en propre ne somme pas à 1.0"

### Ventilation masse salariale

- coûts directs sur activité : projets de recherche en propre (4 types), projets collaboratifs, prestations
- coûts directs hors activité : inter contrat, activités innovation - non ventilables sur projets ni aucun type d'activité
  - typiquement : ingénieurs, techniciens, cdp...
- coûts indirects : fonctionnement général de l'Institut, ont vocation à être ventilés en frais d'environnement sur activités
  - typiquement : fonctions support, business development, management intermédiaire, etc.

In [15]:
# coûts directs sur activités
m.ms_directe_activites = pyo.Var(bounds=(7.0, 13.0))  # REVOIR LES VRAIS CHIFFRES

# coûts directs hors activités
m.ms_directe_intercontrat = pyo.Var(bounds=(0.0, 1.0))
m.ms_directe_innovation = pyo.Var(bounds=(0.0, 0.5))
m.ms_directe_hors_activites = pyo.Expression(
    expr=m.ms_directe_intercontrat + m.ms_directe_innovation
)

# coûts indirects
m.ms_indirects = pyo.Var(bounds=(7.0, 13.0))

# et la somme doit faire la masse salariale totale
m.c_masse_salariale = pyo.Constraint(
    expr=m.ms_directe_activites + m.ms_directe_hors_activites + m.ms_indirects == m.masse_salariale
)

### Ventilation coûts directs activité par types activité

In [16]:
# les coûts directs sur activités se ventilent sur les PRP, projets collaboratifs, prestations
m.ms_directe_prp = pyo.Expression(expr=m.ms_directe_activites * m.beta_prp)
m.ms_directe_projets_collaboratifs = pyo.Expression(expr=m.ms_directe_activites * m.beta_projets_collaboratifs)
m.ms_directe_prestations = pyo.Expression(expr=m.ms_directe_activites * m.beta_prestations)
# NB : on a somme(ms_directe_xx) = m.ms_directe par la contrainte sur le mix d'activité en beta

# ventilation supplémentaire des coûts directs sur PRP par type de projet : RF, RI, DE, FS
m.ms_directe_prp_rf = pyo.Expression(expr=m.ms_directe_prp * m.prp_rf)
m.ms_directe_prp_ri = pyo.Expression(expr=m.ms_directe_prp * m.prp_ri)
m.ms_directe_prp_de = pyo.Expression(expr=m.ms_directe_prp * m.prp_de)
m.ms_directe_prp_fs = pyo.Expression(expr=m.ms_directe_prp * m.prp_fs)
# NB : on a somme(ms_directe_prp_xx) = m.ms_directe_prp par la contrainte sur le mix d'activité

### Ventilation achats et sous-traitance par types activités

In [17]:
# ventilation achats par types activités
m.x_achats_prestations = pyo.Expression(expr=m.x_achats * m.beta_prestations)
m.x_achats_projets_collaboratifs = pyo.Expression(expr=m.x_achats * m.beta_projets_collaboratifs)
m.x_achats_prp = pyo.Expression(expr=m.x_achats * m.beta_prp)

# ventilation supplémentaire des coûts indirects sur PRP par type de projet : RF, RI, DE, FS
m.x_achats_prp_rf = pyo.Expression(expr=m.x_achats_prp * m.prp_rf)
m.x_achats_prp_ri = pyo.Expression(expr=m.x_achats_prp * m.prp_ri)
m.x_achats_prp_de = pyo.Expression(expr=m.x_achats_prp * m.prp_de)
m.x_achats_prp_fs = pyo.Expression(expr=m.x_achats_prp * m.prp_fs)
# NB : on a somme(ms_achats_prp_xx) = m.ms_achats_prp par la contrainte sur le mix d'activité

# ventilation S/T par types activités
m.x_st_prestations = pyo.Expression(expr=m.x_st * m.beta_prestations)
m.x_st_projets_collaboratifs = pyo.Expression(expr=m.x_st * m.beta_projets_collaboratifs)
m.x_st_prp = pyo.Expression(expr=m.x_st * m.beta_prp)

# ventilation supplémentaire des coûts indirects sur PRP par type de projet : RF, RI, DE, FS
m.x_st_prp_rf = pyo.Expression(expr=m.x_st_prp * m.prp_rf)
m.x_st_prp_ri = pyo.Expression(expr=m.x_st_prp * m.prp_ri)
m.x_st_prp_de = pyo.Expression(expr=m.x_st_prp * m.prp_de)
m.x_st_prp_fs = pyo.Expression(expr=m.x_st_prp * m.prp_fs)
# NB : on a somme(ms_st_prp_xx) = m.ms_st_prp par la contrainte sur le mix d'activité

### Ventilation des coûts indirects par types d'activités

In [18]:
# les coûts indirects sur activités se ventilent sur les PRP, projets collaboratifs, prestations
m.ms_indirecte_prp = pyo.Expression(expr=m.ms_indirects * m.beta_prp)
m.ms_indirecte_projets_collaboratifs = pyo.Expression(expr=m.ms_indirects * m.beta_projets_collaboratifs)
m.ms_indirecte_prestations = pyo.Expression(expr=m.ms_indirects * m.beta_prestations)
# NB : on a somme(ms_indirecte_xx) = m.ms_indirecte par la contrainte sur le mix d'activité en beta

# ventilation supplémentaire des coûts directs sur PRP par type de projet : RF, RI, DE, FS
m.ms_indirecte_prp_rf = pyo.Expression(expr=m.ms_indirecte_prp * m.prp_rf)
m.ms_indirecte_prp_ri = pyo.Expression(expr=m.ms_indirecte_prp * m.prp_ri)
m.ms_indirecte_prp_de = pyo.Expression(expr=m.ms_indirecte_prp * m.prp_de)
m.ms_indirecte_prp_fs = pyo.Expression(expr=m.ms_indirecte_prp * m.prp_fs)
# NB : on a somme(ms_indirecte_prp_xx) = m.ms_indirecte_prp par la contrainte sur le mix d'activité

### Coûts environnement

Projets recherche en propre

In [19]:
# 1ERE clef à revoir avec la Finance :
# - 68/7 dans RF ANR 3/7/2025 : applicable ou pas ?
# - répartition MAD suivant le mix d'activités PRP
# - répartition amortissements intégralement dans le portefeuille activités PRP, suivant le mix

# assiettes brutes de calcul...
m.env_prp_rf = pyo.Expression(expr=0.68 * (m.ms_directe_prp_rf + m.mad * m.prp_rf) + 0.07*(m.x_achats_prp_rf + m.x_st_prp_rf + m.amortissements * m.prp_rf)) 
m.env_prp_ri = pyo.Expression(expr=0.68 * (m.ms_directe_prp_ri + m.mad * m.prp_ri) + 0.07*(m.x_achats_prp_ri + m.x_st_prp_ri + m.amortissements * m.prp_ri)) 
m.env_prp_de = pyo.Expression(expr=0.68 * (m.ms_directe_prp_de + m.mad * m.prp_de) + 0.07*(m.x_achats_prp_de + m.x_st_prp_de + m.amortissements * m.prp_de)) 
m.env_prp_fs = pyo.Expression(expr=0.68 * (m.ms_directe_prp_fs + m.mad * m.prp_fs) + 0.07*(m.x_achats_prp_fs + m.x_st_prp_fs + m.amortissements * m.prp_fs)) 
# ... cappées aux indirects
m.c_env_prp_rf = pyo.Constraint(expr=m.env_prp_rf <= m.ms_indirecte_prp_rf + m.structure * m.prp_rf)
m.c_env_prp_ri = pyo.Constraint(expr=m.env_prp_ri <= m.ms_indirecte_prp_ri + m.structure * m.prp_ri)
m.c_env_prp_de = pyo.Constraint(expr=m.env_prp_de <= m.ms_indirecte_prp_de + m.structure * m.prp_de)
m.c_env_prp_fs = pyo.Constraint(expr=m.env_prp_fs <= m.ms_indirecte_prp_fs + m.structure * m.prp_fs)

### Coûts complets

In [20]:
m.couts_complets_prp_rf = pyo.Expression(expr=m.ms_directe_prp_rf + m.mad * m.prp_rf + m.amortissements*m.prp_rf + m.x_achats_prp_rf + m.x_st_prp_rf + m.env_prp_rf)
m.couts_complets_prp_ri = pyo.Expression(expr=m.ms_directe_prp_ri + m.mad * m.prp_ri + m.amortissements*m.prp_ri + m.x_achats_prp_ri + m.x_st_prp_ri + m.env_prp_ri)
m.couts_complets_prp_de = pyo.Expression(expr=m.ms_directe_prp_de + m.mad * m.prp_de + m.amortissements*m.prp_de + m.x_achats_prp_de + m.x_st_prp_de + m.env_prp_de)
m.couts_complets_prp_fs = pyo.Expression(expr=m.ms_directe_prp_fs + m.mad * m.prp_fs + m.amortissements*m.prp_fs + m.x_achats_prp_fs + m.x_st_prp_fs + m.env_prp_fs)

## Intensités de financement (RGEC, RF ANR, etc)

Au premier ordre, Grande Entreprise, pas collab ni large diffusion : A REVOIR ET DISCUTER

In [21]:
# Intensités de financement sur assiette coûts complets PRP
m.int_rf = pyo.Param(initialize=1.00)
m.int_ri = pyo.Param(initialize=0.50)
m.int_de = pyo.Param(initialize=0.25)
m.int_fs = pyo.Param(initialize=0.50)

In [22]:
# Définition des allocations France 2030 socle par type de projets
m.f2030_rf = pyo.Var(within=pyo.NonNegativeReals)
m.f2030_ri = pyo.Var(within=pyo.NonNegativeReals)
m.f2030_de = pyo.Var(within=pyo.NonNegativeReals)
m.f2030_fs = pyo.Var(within=pyo.NonNegativeReals)
# ... cappées aux intensités de financement
m.c_f2030_rf = pyo.Constraint(expr=m.f2030_rf <= m.int_rf * m.couts_complets_prp_rf)
m.c_f2030_ri = pyo.Constraint(expr=m.f2030_ri <= m.int_ri * m.couts_complets_prp_ri)
m.c_f2030_de = pyo.Constraint(expr=m.f2030_de <= m.int_de * m.couts_complets_prp_de)
m.c_f2030_fs = pyo.Constraint(expr=m.f2030_fs <= m.int_fs * m.couts_complets_prp_fs)
# et sommant à la dotation totale
m.c_f2030 = pyo.Constraint(expr=m.f2030_rf + m.f2030_ri + m.f2030_de + m.f2030_fs == m.f2030)

### Projets collaboratifs et prestations

In [23]:
# Intensités de financement sur assiette coûts complets pour projets collaboratifs (HE, AID, etc)
m.int_projets_collaboratifs = pyo.Param(initialize=0.25)

# calcul des coûts environnement (NB : voir avec la Finance si on peut prendre ou pas une part d'amortissements)
m.env_projets_collaboratifs = pyo.Expression(
    expr=m.int_projets_collaboratifs * (m.ms_directe_projets_collaboratifs + m.x_achats_projets_collaboratifs + m.x_st_projets_collaboratifs)
)

# couts complets
m.couts_complets_projets_collaboratifs = pyo.Expression(expr=m.ms_directe_projets_collaboratifs + m.env_projets_collaboratifs)

# revenus projets collaboratifs
m.c_projets_collaboratifs_min = pyo.Constraint(
    expr=m.projets_collaboratifs == (1 + m.int_projets_collaboratifs) * m.couts_complets_projets_collaboratifs
)

In [24]:
# objectif de marge sur prestations
m.obj_marge_prestations = pyo.Param(initialize=0.30)

# taux environnement sur prestations
m.taux_env_prestations = pyo.Param(initialize=0.52)

# calcul des coûts environnement (NB : voir avec la Finance si on peut prendre ou pas une part d'amortissements)
m.env_prestations = pyo.Expression(
    expr=m.taux_env_prestations * (m.ms_directe_prestations + m.x_achats_prestations + m.x_st_prestations)
)

# couts complets
m.couts_complets_prestations = pyo.Expression(expr=m.ms_directe_prestations + m.env_prestations)

# revenus prestations
m.c_prestations_min = pyo.Constraint(
    expr=m.prestations >= (1 + m.obj_marge_prestations) * m.couts_complets_prestations
)
m.c_prestations_max = pyo.Constraint(
    expr=m.prestations <= (1 + 1.5*m.obj_marge_prestations) * m.couts_complets_prestations
)

# Régles

## Convention

In [25]:
# 1. Dotation France 2030 Socle cappée à 11 M€ : voir plus haut
# m.f2030 = pyo.Var(bounds=(0.0, 11.0))  

In [26]:
# 2. Dotation France 2030 inférieure au tiers des dépenses (comptables) de l'Institut
m.c_f2030_max = pyo.Constraint(
    expr=m.f2030 <= (1/3) * m.charges
)

In [27]:
# 3. Cofinancements privés projets de recherche en propre >= France 2030 socle
m.c_cofinancements = pyo.Constraint(
    expr=m.cofinancements_prives >= m.f2030
)

### Sous-traitance

In [28]:
# 4. part maximale de sous-traitance dans les projets de recherche en propre

m.c_st_rf = pyo.Constraint(expr=m.x_st_prp_rf <= 0.50*m.couts_complets_prp_rf)
m.c_st_ri = pyo.Constraint(expr=m.x_st_prp_ri <= 0.50*m.couts_complets_prp_ri)
m.c_st_de = pyo.Constraint(expr=m.x_st_prp_de <= 0.50*m.couts_complets_prp_de)
m.c_st_fs = pyo.Constraint(expr=m.x_st_prp_fs <= 0.50*m.couts_complets_prp_fs)

# Objectif

In [29]:
# on introduit une variable slack >= 0, la contrainte resultat + slack >= 0, et on va minimiser slack
#
# si la valeur slack=0 est atteinte, cela veut dire qu'il existe un m.resultat >= 0. 
# L'Institut est viable sous les contraintes et intervalles de variables donnés
#
# si la valeur optimale est slack>0, cela veut dire que le meilleur resultat est négatif. 
# slack mesure l'écart comptable à l'équilibre.

m.slack_variable = pyo.Var(within=pyo.NonNegativeReals)

m.c_resultat = pyo.Constraint(expr=m.resultat + m.slack_variable >= 0.0)

m.objectif = pyo.Objective(expr=m.slack_variable, sense=pyo.minimize)

### Vérifications diverses

In [30]:
# 1. Check structure
# m.display()

# # 2. Write LP file (best consistency check)
# try:
#     m.write('model_check.lp')
#     print("Model is syntactically valid")
# except Exception as e:
#     print(f"Model error: {e}")

In [31]:
# for expr in m.component_data_objects(pyo.Expression):
#     print(f"{expr.name}: {expr.expr}")

### Résolution

In [32]:
# Solve the model
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(m, load_solutions=True)
# Show solver report
results.write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 0.0
  Upper bound: 0.0
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal


In [33]:
# m.write('debug.lp', io_options={'symbolic_solver_labels': True})

### Résultats

In [34]:
def print_results(m):
    """Print model results with detailed financial breakdown."""

    print("="*70)
    print("FINANCIAL MODEL RESULTS (M€)")
    print("="*70)

    # 1. MACRO INPUTS
    print("\n1. MACRO INPUTS (Parameters)")
    print("-"*70)
    for para in m.component_data_objects(pyo.Param):
        print(f"{para.name:40s} = {pyo.value(para):.2f}")

    # 2. MACRO OUTPUTS
    print("\n2. MACRO OUTPUTS")
    print("-"*70)
    print("\n   Vue comptable:")

    # Produits breakdown and total
    print("\n      Produits:")
    produits_components = ['mad', 'cash', 'f2030', 'prestations', 'projets_collaboratifs', 'autres_revenus']
    for name in produits_components:
        if hasattr(m, name):
            print(f"         {name:30s} = {pyo.value(getattr(m, name)):6.2f}")
    if hasattr(m, 'produits'):
        print(f"         {'Produits total':30s} = {pyo.value(m.produits):6.2f}")

    # Charges breakdown and total
    print("\n      Charges:")
    charges_components = ['masse_salariale', 'x_achats', 'x_st', 'mad', 'structure', 'amortissements', 'frais_fi', 'provisions']
    for name in charges_components:
        if hasattr(m, name):
            print(f"         {name:30s} = {pyo.value(getattr(m, name)):6.2f}")
    if hasattr(m, 'charges'):
        print(f"         {'Charges total':30s} = {pyo.value(m.charges):6.2f}")

    # Resultat
    if hasattr(m, 'resultat'):
        print(f"         {'Resultat':30s} = {pyo.value(m.resultat):6.2f}")

    # 3. VIEW BY PROJECT TYPE
    print("\n3. VIEW BY PROJECT TYPE")
    print("-"*70)

    # PRP types: RF, RI, DE, FS
    prp_types = ['rf', 'ri', 'de', 'fs']

    for ptype in prp_types:
        print(f"\n   {ptype.upper()}:")

        # Revenue section
        if hasattr(m, f'prp_{ptype}'):
            prp_val = pyo.value(getattr(m, f'prp_{ptype}'))
            mad_rev = pyo.value(m.mad) * prp_val
            cash_rev = pyo.value(m.cash) * prp_val
            f2030_rev = pyo.value(getattr(m, f'f2030_{ptype}')) if hasattr(m, f'f2030_{ptype}') else 0
            total_rev = mad_rev + cash_rev + f2030_rev
            print(f"      Total Revenues:        {total_rev:8.2f}")
            print(f"         mad           {mad_rev:8.2f}")
            print(f"         cash          {cash_rev:8.2f}")
            print(f"         f2030         {f2030_rev:8.2f}")

        # Charges section
        charge_items = [
            ('ms_directe', f'ms_directe_prp_{ptype}'),
            ('achats', f'x_achats_prp_{ptype}'),
            ('st', f'x_st_prp_{ptype}'),
            ('ms_indirects', f'ms_indirecte_prp_{ptype}')
        ]
        total_charges = 0
        for label, var_name in charge_items:
            if hasattr(m, var_name):
                val = pyo.value(getattr(m, var_name))
                total_charges += val
                # Print total charges with computed value
        print(f"      Total Charges:        {total_charges:8.2f}")
        for label, var_name in charge_items:
            if hasattr(m, var_name):
                val = pyo.value(getattr(m, var_name))
                print(f"         {label:14s}{val:8.2f}")


    # Projets collaboratifs
    print("\n   PROJETS COLLABORATIFS:")
    total_rev_collab = pyo.value(m.projets_collaboratifs) if hasattr(m, 'projets_collaboratifs') else 0
    print(f"      Total Revenues:        {total_rev_collab:8.2f}")
    print("      Costs:")
    total_charges_collab = 0
    collab_costs = ['ms_directe_projets_collaboratifs', 'x_achats_projets_collaboratifs', 'x_st_projets_collaboratifs']
    for name in collab_costs:
        if hasattr(m, name):
            val = pyo.value(getattr(m, name))
            total_charges_collab += val
            print(f"         {name:30s} = {val:8.2f}")
    print(f"      Total Charges:        {total_charges_collab:8.2f}")

    # Prestations
    print("\n   PRESTATIONS:")
    total_rev_prest = pyo.value(m.prestations) if hasattr(m, 'prestations') else 0
    print(f"      Total Revenues:        {total_rev_prest:8.2f}")
    print("      Costs:")
    total_charges_prest = 0
    prest_costs = ['ms_directe_prestations', 'x_achats_prestations', 'x_st_prestations']
    for name in prest_costs:
        if hasattr(m, name):
            val = pyo.value(getattr(m, name))
            total_charges_prest += val
            print(f"         {name:30s} = {val:8.2f}")
    print(f"      Total Charges:        {total_charges_prest:8.2f}")

In [35]:
print_results(m)

FINANCIAL MODEL RESULTS (M€)

1. MACRO INPUTS (Parameters)
----------------------------------------------------------------------
f2030_max                                = 11.00
structure                                = 6.00
amortissements                           = 0.00
frais_fi                                 = 0.00
provisions                               = 0.00
beta_prp                                 = 0.70
beta_projets_collaboratifs               = 0.05
beta_prestations                         = 0.25
prp_rf                                   = 0.05
prp_ri                                   = 0.95
prp_de                                   = 0.00
prp_fs                                   = 0.00
int_rf                                   = 1.00
int_ri                                   = 0.50
int_de                                   = 0.25
int_fs                                   = 0.50
int_projets_collaboratifs                = 0.25
obj_marge_prestations                    = 0.30
taux_